In [ ]:
import pathlib

import astropy.table as at
import astropy.units as u
import numpy as np
from astropy.time import Time
from astropy.units import Quantity as Q

In [ ]:
epoch_astrometry_file = pathlib.Path(
    "~/data/Gaia/DR4/gaia-dr4-prerelease-epoch-astrometry_2026-06-26/GAIA_DR4_PRERELEASE_EPOCH_ASTROMETRY_RAW.xml"
)
table = at.Table.read(epoch_astrometry_file, format="votable")
first_mask = ~table["source_id"].mask

table = table[first_mask]

In [ ]:
TCB_REFERENCE_EPOCH = Time("2010-01-01T00:00:00", format="isot", scale="tcb")
DR4_REFERENCE_EPOCH = Time("2017.5", format="jyear", scale="tcb")

table["relative_time_day"] = (
    TCB_REFERENCE_EPOCH.jyear
    + table["obs_time_tcb"] * (u.nanosecond.to(u.year))
    + table["obs_time_bary_corr"] * (u.nanosecond.to(u.year))
    - DR4_REFERENCE_EPOCH.jyear
) * u.year.to(u.day)

In [ ]:
source_ids = np.unique(table["source_id"])

In [ ]:
source_id_to_name = {
    1457486023639239296: "Gaia 4",
    4318465066420528000: "Gaia BH3",
    3937211745905473024: "HD 114762",
    435469040545191680: "plx G14",
    3926186255616949504: "plx G19",
    4181040337841125632: "plx G9",
    2237987199365376: "qso 1",
    10973744521070720: "qso 2",
    60730287810150016: "qso 3",
}

# parallax in mas, period in days
truths = {
    "Gaia 4": {"parallax": 13.643, "period": 571.3},
    "Gaia BH3": {"parallax": 1.675, "period": 4194.7},  # from astrometric solution only
    "HD 114762": {"parallax": 24.855, "period": 83.9},
}

Collapse CCD measurements into a single measurement per transit:

In [ ]:
def get_data(rows):
    preproc_time_day = np.array(
        [
            np.average(
                row["relative_time_day"][1:][row["used_by_agis_al"][1:].filled(False)]
            )
            for row in rows
        ]
    )
    preproc_pos_al = np.array(
        [
            np.average(
                row["centroid_pos_al"][1:][row["used_by_agis_al"][1:].filled(False)],
                weights=1
                / row["centroid_pos_error_al"][1:][
                    row["used_by_agis_al"][1:].filled(False)
                ]
                ** 2,
            )
            for row in rows
        ]
    )
    preproc_pos_al_err = np.array(
        [
            np.sqrt(
                1
                / np.sum(
                    1
                    / row["centroid_pos_error_al"][1:][
                        row["used_by_agis_al"][1:].filled(False)
                    ]
                    ** 2
                )
            )
            for row in rows
        ]
    )

    # preproc_pos_al_err = np.sqrt(
    #     preproc_pos_al_err**2 + 0.1**2
    # )

    excess_noise = np.array(rows["agis_source_excess_noise"].filled(np.nan))

    preproc_scan_angle = np.array(
        [
            np.average(
                row["scan_pos_angle"][1:][row["used_by_agis_al"][1:].filled(False)]
            )
            for row in rows
        ]
    )

    preproc_parallax_factor = np.array(rows["parallax_factor_al"])

    _mask = (
        np.isfinite(preproc_time_day)
        & np.isfinite(preproc_pos_al)
        & np.isfinite(preproc_pos_al_err)
        & np.isfinite(preproc_scan_angle)
        & np.isfinite(preproc_parallax_factor)
    )
    print(_mask.sum(), len(_mask))

    return {
        "relative_time": Q(preproc_time_day[_mask].astype("f8"), "day"),
        "pos_al": Q(preproc_pos_al[_mask].astype("f8"), "mas"),
        "pos_al_err": Q(preproc_pos_al_err[_mask].astype("f8"), "mas"),
        "scan_angle": Q(preproc_scan_angle[_mask].astype("f8"), "deg"),
        "parallax_factor": preproc_parallax_factor[_mask].astype("f8"),
        "excess_noise": Q(excess_noise[_mask].astype("f8"), "mas"),
    }

In [ ]:
this_path = pathlib.Path(".").resolve()
dr4_preview_path = this_path / "gaia-dr4-prerelease"
dr4_preview_path.mkdir(exist_ok=True)

In [ ]:
for source_id in source_ids:
    rows = table[table["source_id"] == source_id]
    name = source_id_to_name.get(source_id, str(source_id)).replace(" ", "-")
    print(source_id, name)

    data = at.QTable(get_data(rows))

    # put parallax, period into table metadata
    data.meta["parallax_mas"] = truths.get(name, {}).get("parallax", np.nan)
    data.meta["period_day"] = truths.get(name, {}).get("period", np.nan)

    data.write(dr4_preview_path / f"{name}.ecsv", overwrite=True)